# Perceptrón simple: implementación desde cero con NumPy

**Materiales desarrollados por Matías Barreto, 2025**

**Tecnicatura en Ciencia de Datos - IFTS**

**Asignatura:** Procesamiento de Lenguaje Natural

---

## Objetivo

Construir un perceptrón simple desde cero para ver, paso a paso, cómo una neurona artificial transforma texto vectorizado en una decisión binaria.

## Resultados de aprendizaje

Al final de este notebook vas a poder:

1. Explicar qué partes componen un perceptrón.
2. Implementar una función de activación escalón.
3. Seguir la regla de actualización de pesos cuando el modelo se equivoca.
4. Interpretar los pesos aprendidos como una señal de qué palabras empujan una decisión.
5. Reconocer por qué el perceptrón simple no alcanza para problemas más complejos.

## Relación con los notebooks anteriores

En `01` y `02` trabajaste con modelos clásicos ya implementados. Ahora mantenemos la idea de representar texto como números, pero cambiamos el foco: en vez de usar una caja ya armada, vamos a programar una neurona artificial a mano.

## Introducción

El perceptrón fue una de las primeras implementaciones operativas de una neurona artificial. Aunque hoy se usa sobre todo con valor histórico y didáctico, sigue siendo un excelente punto de entrada para entender qué hace una red neuronal antes de pasar a PyTorch, Keras o HuggingFace.


---

## 1. Importación de Librerías

Solo necesitamos NumPy para operaciones numéricas. No vamos a usar scikit-learn ni ningún framework de ML.

In [ ]:
# NumPy: Para arrays y operaciones numéricas
import numpy as np

# Matplotlib: Para visualización (opcional)
import matplotlib.pyplot as plt

# Fijamos semilla para reproducibilidad
# Esto garantiza que los pesos aleatorios iniciales sean siempre los mismos
np.random.seed(42)

print("Librerías importadas correctamente.")
print(f"NumPy versión: {np.__version__}")

---

## 2. Dataset: análisis de sentimiento en español rioplatense

Vamos a trabajar con un corpus chico y completamente visible. El objetivo no es entrenar un modelo competitivo, sino mirar con claridad cómo una frase pasa a un vector y cómo ese vector modifica los pesos del perceptrón.


In [ ]:
# Corpus de frases con sentimiento etiquetado
# 1 = Positivo, 0 = Negativo
frases = [
    "Amo el verano en Buenos Aires",
    "No me gusta el tráfico matutino",
    "Este asado está espectacular",
    "Qué bajón, perdí el colectivo",
    "Me encanta salir los domingos",
    "Detesto el calor húmedo"
]

# Etiquetas: 1 = sentimiento positivo, 0 = sentimiento negativo
etiquetas = np.array([1, 0, 1, 0, 1, 0])

print("Corpus de entrenamiento:")
print("=" * 70)
for i in range(len(frases)):
    frase = frases[i]
    etiq = etiquetas[i]
    sentimiento = "POSITIVO" if etiq == 1 else "NEGATIVO"
    print(f"{i + 1}. [{sentimiento}] {frase}")

print(f"Total de frases: {len(frases)}")
print(f"Distribución: {np.bincount(etiquetas)} (negativas, positivas)")


---

## 3. Construcción del Vocabulario

Definimos manualmente un vocabulario de palabras clave con carga emocional. En un sistema real, esto se construiría automáticamente, pero aquí lo hacemos explícito para fines didácticos.

In [ ]:
# Vocabulario: palabras con carga emocional
# Elegimos palabras que aparecen en las frases y que tienen connotación clara
vocabulario = [
    "amo",           # Positiva
    "no",            # Negativa (negación)
    "gusta",         # Positiva
    "asado",         # Positiva (cultural argentino)
    "espectacular",  # Positiva
    "bajón",         # Negativa (jerga argentina)
    "perdí",         # Negativa
    "encanta",       # Positiva
    "detesto",       # Negativa
    "calor"          # Neutral/contexto
]

print(f"Vocabulario construido con {len(vocabulario)} palabras:")
print(vocabulario)

print("\nNota: Este vocabulario es pequeño y manual por fines pedagógicos.")
print("En un sistema real, usaríamos TF-IDF o embeddings pre-entrenados.")

---

## 4. Vectorización: De Texto a Números

Convertimos cada frase en un **vector binario** de longitud igual al vocabulario:
- `1` si la palabra aparece en la frase
- `0` si no aparece

Esto se conoce como **Bag of Words binario** (ignora frecuencia, solo presencia/ausencia).

In [ ]:
def vectorizar(frase, vocabulario):
    """
    Convierte una frase en un vector binario según el vocabulario.

    Args:
        frase (str): La frase a vectorizar.
        vocabulario (list): Lista de palabras del vocabulario.

    Returns:
        np.array: Vector binario de longitud len(vocabulario).
    """
    # Pasamos la frase a minúsculas y la partimos en tokens simples.
    tokens = frase.lower().split()

    # Recorremos el vocabulario palabra por palabra.
    vector_valores = []
    for palabra in vocabulario:
        if palabra in tokens:
            vector_valores.append(1)
        else:
            vector_valores.append(0)

    vector = np.array(vector_valores)
    return vector


# Aplicamos la vectorización a todas las frases.
filas_vectorizadas = []
for frase in frases:
    fila_vectorizada = vectorizar(frase, vocabulario)
    filas_vectorizadas.append(fila_vectorizada)

X = np.array(filas_vectorizadas)

print("Matriz de características (X):")
print("=" * 70)
print(f"Forma: {X.shape} (filas = frases, columnas = palabras del vocabulario)")
print("
Primeras 3 filas:")
print(X[:3])

print("
Interpretación de la primera frase:")
print(f"Frase: '{frases[0]}'")
print(f"Vector: {X[0]}")

palabras_presentes = []
for indice_palabra in range(len(vocabulario)):
    if X[0][indice_palabra] == 1:
        palabras_presentes.append(vocabulario[indice_palabra])

print(f"Palabras detectadas: {palabras_presentes}")


---

## 5. Arquitectura del Perceptrón

Un perceptrón es una neurona artificial que realiza una clasificación binaria. Su arquitectura consta de:

### Componentes:

1. **Vector de entrada (x)**: Las features (en nuestro caso, el vector de palabras)
2. **Pesos sinápticos (w)**: Un peso por cada feature, determina la importancia
3. **Bias (b)**: Un término constante (sesgo)
4. **Suma ponderada (z)**: z = w₁x₁ + w₂x₂ + ... + wₙxₙ + b
5. **Función de activación (σ)**: Transforma z en una predicción

### Fórmula matemática:

$$y = \sigma(\sum_{i=1}^{n} w_i x_i + b)$$

Donde:
- $x_i$: Feature i
- $w_i$: Peso de la feature i
- $b$: Bias
- $\sigma$: Función de activación escalón

### Función de activación escalón:

$$
\sigma(z) =
\begin{cases}
1 & \text{si } z > 0 \\
0 & \text{si } z \leq 0
\end{cases}
$$

### Visualización:

```
x₁ ──w₁──┐
x₂ ──w₂──┤
x₃ ──w₃──├──> Σ + b ──> σ(z) ──> y (0 o 1)
   ...   │
xₙ ──wₙ──┘
```

In [ ]:
# Inicializamos los parámetros del perceptrón

# Dimensión del vector de entrada (número de palabras en el vocabulario)
n_features = len(vocabulario)

# Pesos sinápticos: uno por cada feature
# Inicialización aleatoria desde una distribución normal
# randn() genera números con media 0 y desviación estándar 1
pesos = np.random.randn(n_features)

# Bias (sesgo): inicializado en 0
bias = 0.0

print("Parámetros del perceptrón inicializados:")
print("=" * 70)
print(f"Número de features: {n_features}")
print(f"\nPesos iniciales (aleatorios):")
for i, palabra in enumerate(vocabulario):
    print(f"  w[{palabra}] = {pesos[i]:.4f}")
print(f"\nBias inicial: {bias}")

print("\nNota: Los pesos aleatorios se ajustarán durante el entrenamiento.")

---

## 6. Función de Activación: Escalón

La función escalón (step function) es la más simple de las funciones de activación:
- Retorna `1` si la entrada es positiva
- Retorna `0` si la entrada es negativa o cero

**Limitación importante**: No es diferenciable en z=0, por lo que no se puede usar con backpropagation. Por eso las redes modernas usan ReLU, sigmoid o tanh.

In [ ]:
def activacion_escalon(z):
    """
    Función de activación escalón.
    """
    return 1 if z > 0 else 0


z_values = np.linspace(-5, 5, 100)
activaciones = []
for z in z_values:
    activaciones.append(activacion_escalon(z))

plt.figure(figsize=(10, 4))
plt.plot(z_values, activaciones, linewidth=2, color='blue')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z (suma ponderada)', fontsize=12)
plt.ylabel('σ(z) (salida)', fontsize=12)
plt.title('Función de activación escalón', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print("Características de la función escalón:")
print("- Discontinua en z=0")
print("- No diferenciable (derivada no definida en z=0)")
print("- Salida binaria: solo 0 o 1")
print("- Históricamente importante, pero desplazada por activaciones diferenciables en redes modernas")


---

## 7. Función de Predicción

Implementamos la función forward (propagación hacia adelante) del perceptrón.

In [ ]:
def predecir(x, pesos, bias):
    """
    Realiza una predicción con el perceptrón.

    Args:
        x (np.array): Vector de entrada (features)
        pesos (np.array): Vector de pesos
        bias (float): Término de sesgo

    Returns:
        int: Predicción (0 o 1)

    Proceso:
        1. Calcular suma ponderada: z = w·x + b
        2. Aplicar función de activación: y = σ(z)
    """
    # Producto punto: suma de productos elemento a elemento
    # np.dot(w, x) = w₁x₁ + w₂x₂ + ... + wₙxₙ
    z = np.dot(pesos, x) + bias

    # Aplicamos la función de activación
    y_pred = activacion_escalon(z)

    return y_pred


# Probamos la función de predicción con una frase de ejemplo
print("Prueba de predicción con pesos aleatorios (antes del entrenamiento):")
print("=" * 70)

frase_test = frases[0]
x_test = X[0]
y_real = etiquetas[0]

prediccion = predecir(x_test, pesos, bias)

print(f"Frase: '{frase_test}'")
print(f"Etiqueta real: {y_real} ({'Positivo' if y_real == 1 else 'Negativo'})")
print(f"Predicción: {prediccion} ({'Positivo' if prediccion == 1 else 'Negativo'})")
print(f"\n¿Es correcta? {prediccion == y_real}")
print("\nNota: Es probable que falle porque los pesos son aleatorios.")
print("El entrenamiento ajustará los pesos para mejorar las predicciones.")

---

## 8. Regla de Aprendizaje del Perceptrón

El perceptrón aprende ajustando sus pesos cuando comete un error. La regla de actualización es:

### Fórmula de actualización:

$$w_i^{nuevo} = w_i^{viejo} + \alpha \cdot error \cdot x_i$$

$$b^{nuevo} = b^{viejo} + \alpha \cdot error$$

Donde:
- $\alpha$: Tasa de aprendizaje (learning rate), controla el tamaño del paso
- $error = y_{real} - y_{predicho}$: Diferencia entre lo esperado y lo predicho
- $x_i$: Valor de la feature i

### Intuición:

**Si error = 0** (predicción correcta):
- No se actualizan los pesos

**Si error = +1** (debía predecir 1 pero predijo 0):
- Los pesos aumentan proporcionalmente a las features activas
- Esto hace que la próxima vez sea más probable predecir 1

**Si error = -1** (debía predecir 0 pero predijo 1):
- Los pesos disminuyen proporcionalmente a las features activas
- Esto hace que la próxima vez sea más probable predecir 0

Esta regla es la versión más simple de **descenso de gradiente**, el algoritmo fundamental del deep learning.

In [ ]:
def entrenar_perceptron(X, y, tasa_aprendizaje=0.1, epocas=20, verbose=True):
    """
    Entrena un perceptrón simple usando la regla de aprendizaje del perceptrón.

    Args:
        X (np.array): Matriz de características (n_muestras, n_features)
        y (np.array): Vector de etiquetas (n_muestras,)
        tasa_aprendizaje (float): Tamaño del paso de actualización (α)
        epocas (int): Número de pasadas completas por el dataset
        verbose (bool): Si True, imprime progreso del entrenamiento

    Returns:
        tuple: (pesos_finales, bias_final, historial_errores)
    """
    # Inicialización de parámetros
    n_muestras, n_features = X.shape
    pesos = np.random.randn(n_features)
    bias = 0.0

    # Lista para guardar el número de errores por época
    historial_errores = []

    if verbose:
        print("Iniciando entrenamiento del perceptrón...")
        print("=" * 70)
        print(f"Tasa de aprendizaje (α): {tasa_aprendizaje}")
        print(f"Épocas: {epocas}")
        print(f"Muestras de entrenamiento: {n_muestras}")
        print("=" * 70)
        print()

    # Bucle de entrenamiento
    for epoca in range(epocas):
        errores = 0  # Contador de errores en esta época

        # Iteramos sobre cada muestra de entrenamiento
        for i in range(n_muestras):
            x_i = X[i]      # Vector de entrada i
            y_real = y[i]   # Etiqueta real i

            # Paso 1: Hacer predicción
            y_pred = predecir(x_i, pesos, bias)

            # Paso 2: Calcular error
            error = y_real - y_pred

            # Paso 3: Actualizar pesos si hay error
            if error != 0:
                # Actualización de pesos: w = w + α × error × x
                pesos += tasa_aprendizaje * error * x_i

                # Actualización de bias: b = b + α × error
                bias += tasa_aprendizaje * error

                # Incrementamos contador de errores
                errores += 1

        # Guardamos el número de errores de esta época
        historial_errores.append(errores)

        # Imprimimos progreso
        if verbose:
            print(f"Época {epoca + 1:2d}/{epocas}: Errores = {errores}")

    if verbose:
        print("\n" + "=" * 70)
        print("Entrenamiento completado.")
        if errores == 0:
            print("El modelo convergió: no hubo errores en la última época.")
        else:
            print(f"Nota: Quedaron {errores} errores. El problema podría no ser linealmente separable.")

    return pesos, bias, historial_errores


# Ejecutamos el entrenamiento
pesos_entrenados, bias_entrenado, historial = entrenar_perceptron(
    X, etiquetas,
    tasa_aprendizaje=0.1,
    epocas=20,
    verbose=True
)

---

## 9. Curva de aprendizaje

Ahora miramos cuántos errores comete el perceptrón en cada época. Esta curva no reemplaza una evaluación externa, pero sí permite ver si el algoritmo está corrigiendo errores a medida que actualiza sus pesos.


In [ ]:
# Gráfico de errores por época
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(historial) + 1), historial, marker='o', linewidth=2, markersize=8)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Número de Errores', fontsize=12)
plt.title('Curva de Aprendizaje del Perceptrón', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(historial) + 1))
plt.tight_layout()
plt.show()

print("Interpretación de la curva:")
print("- Una curva descendente indica que el modelo está aprendiendo")
print("- Si llega a 0, el modelo convergió (clasifica perfectamente el training set)")
print("- Si oscila sin bajar, el problema podría no ser linealmente separable")

---

## 10. Análisis de Pesos Aprendidos

Inspeccionamos los pesos finales para entender qué aprendió el modelo.

In [ ]:
print("Pesos aprendidos por el perceptrón:")
print("=" * 70)

# Ordenamos palabras por peso (de más positivo a más negativo)
indices_ordenados = np.argsort(pesos_entrenados)[::-1]

print("\nPalabras ordenadas por influencia (de más positiva a más negativa):\n")
for idx in indices_ordenados:
    palabra = vocabulario[idx]
    peso = pesos_entrenados[idx]
    if peso > 0:
        influencia = "POSITIVA"
    elif peso < 0:
        influencia = "NEGATIVA"
    else:
        influencia = "NEUTRA"

    print(f"  {palabra:15s} → peso = {peso:7.4f}  [{influencia}]")

print(f"\nBias (sesgo): {bias_entrenado:.4f}")

print("\n" + "=" * 70)
print("Interpretación:")
print("- Pesos positivos: Palabras asociadas con sentimiento positivo")
print("- Pesos negativos: Palabras asociadas con sentimiento negativo")
print("- Magnitud del peso: Importancia de la palabra en la decisión")

---

## 11. Dos lecturas necesarias del resultado

Primero vamos a mirar qué pasa sobre las frases con las que el modelo aprendió. Después vamos a hacer una prueba externa muy pequeña con frases nuevas y etiquetadas a mano.

Con un corpus tan chico, esta segunda lectura no es una evaluación rigurosa. Sí funciona como advertencia pedagógica: acertar en entrenamiento no significa generalizar bien.


In [ ]:
print("Control sobre las frases de entrenamiento:")
print("=" * 70)

aciertos_entrenamiento = 0
for i in range(len(frases)):
    x_i = X[i]
    y_real = etiquetas[i]
    y_pred = predecir(x_i, pesos_entrenados, bias_entrenado)

    if y_pred == y_real:
        aciertos_entrenamiento += 1
        marca = "OK"
    else:
        marca = "REVISAR"

    sentimiento_real = "Positivo" if y_real == 1 else "Negativo"
    sentimiento_pred = "Positivo" if y_pred == 1 else "Negativo"

    print(f"
{marca} Frase: '{frases[i]}'")
    print(f"  Real: {sentimiento_real} | Predicción: {sentimiento_pred}")

accuracy_entrenamiento = aciertos_entrenamiento / len(frases)
print("
" + "=" * 70)
print(f"Accuracy de entrenamiento: {aciertos_entrenamiento}/{len(frases)} = {accuracy_entrenamiento:.2%}")

frases_prueba_etiquetadas = [
    "No me gustó el lugar, fue una decepción.",
    "La atención fue excelente y volvería sin dudarlo.",
    "Qué bajón perder tanto tiempo en el tráfico.",
    "Me encanta salir a caminar cuando baja el calor."
]

etiquetas_prueba = np.array([0, 1, 0, 1])

print("
Prueba externa pequeña:")
print("=" * 70)

aciertos_prueba = 0
for i in range(len(frases_prueba_etiquetadas)):
    frase_prueba = frases_prueba_etiquetadas[i]
    etiqueta_real = etiquetas_prueba[i]
    vector_prueba = vectorizar(frase_prueba, vocabulario)
    prediccion_prueba = predecir(vector_prueba, pesos_entrenados, bias_entrenado)

    if prediccion_prueba == etiqueta_real:
        aciertos_prueba += 1
        marca = "OK"
    else:
        marca = "REVISAR"

    sentimiento_real = "Positivo" if etiqueta_real == 1 else "Negativo"
    sentimiento_pred = "Positivo" if prediccion_prueba == 1 else "Negativo"

    print(f"
{marca} Frase nueva: '{frase_prueba}'")
    print(f"  Real esperado: {sentimiento_real} | Predicción: {sentimiento_pred}")

accuracy_prueba = aciertos_prueba / len(frases_prueba_etiquetadas)
print("
" + "=" * 70)
print(f"Accuracy en prueba externa: {aciertos_prueba}/{len(frases_prueba_etiquetadas)} = {accuracy_prueba:.2%}")
print("
Lectura pedagógica: si el rendimiento cae fuera del entrenamiento, el modelo está generalizando peor de lo que parecía.")


---

## 12. Predicción sobre frases nuevas sin etiqueta

Después del control anterior, podemos usar el perceptrón como herramienta exploratoria sobre frases nuevas. En este bloque ya no hay una respuesta correcta cargada: el objetivo es mirar qué palabras reconoce y qué tan fuerte es la suma ponderada.


In [ ]:
# Frases nuevas para explorar
frases_prueba = [
    "No aguanto este calor",
    "Qué hermoso día para pasear",
    "Detesto levantarme temprano",
    "Me encanta el asado del domingo",
    "Qué bajón el tráfico de hoy"
]

print("Predicciones sobre frases nuevas:")
print("=" * 70)

for frase in frases_prueba:
    x_test = vectorizar(frase, vocabulario)
    prediccion = predecir(x_test, pesos_entrenados, bias_entrenado)
    z = np.dot(pesos_entrenados, x_test) + bias_entrenado

    sentimiento = "POSITIVO" if prediccion == 1 else "NEGATIVO"

    palabras_detectadas = []
    for indice_palabra in range(len(vocabulario)):
        if x_test[indice_palabra] == 1:
            palabras_detectadas.append(vocabulario[indice_palabra])

    print(f"
Frase: '{frase}'")
    print(f"Predicción: {sentimiento}")
    print(f"Suma ponderada (z): {z:.4f}")
    print(f"Palabras detectadas: {palabras_detectadas}")


---

## 13. Limitaciones del Perceptrón Simple

El perceptrón tiene restricciones importantes que debemos entender.

In [ ]:
print("LIMITACIONES DEL PERCEPTRÓN SIMPLE")
print("=" * 70)

print("\n1. SOLO PROBLEMAS LINEALMENTE SEPARABLES")
print("-" * 70)
print("El perceptrón solo puede aprender patrones que se pueden separar con")
print("una línea recta (en 2D) o un hiperplano (en >2D).")
print("\nEjemplo clásico: No puede resolver XOR (Minsky & Papert, 1969)")
print("\nXOR:  A  B  | Salida")
print("      0  0  |   0")
print("      0  1  |   1")
print("      1  0  |   1")
print("      1  1  |   0")
print("\nNo existe una línea recta que separe correctamente estos 4 puntos.")

print("\n2. IGNORA EL ORDEN DE LAS PALABRAS")
print("-" * 70)
print("Bag of Words trata 'No me gusta' igual que 'Me gusta, no'")
print("El modelo no captura negaciones ni contexto.")

print("\n3. NO CAPTURA RELACIONES COMPLEJAS")
print("-" * 70)
print("Solo aprende combinaciones lineales de features.")
print("No puede modelar interacciones entre palabras (ej: 'no muy bueno').")

print("\n4. FUNCIÓN DE ACTIVACIÓN NO DIFERENCIABLE")
print("-" * 70)
print("La función escalón no tiene derivada en z=0.")
print("Esto impide usar backpropagation para redes multicapa.")

print("\n5. SENSIBLE A OUTLIERS Y RUIDO")
print("-" * 70)
print("Un solo ejemplo mal etiquetado puede impedir convergencia.")

print("\n" + "=" * 70)
print("SOLUCIÓN: Redes Neuronales Multicapa (MLP)")
print("=" * 70)
print("Agregar capas ocultas permite:")
print("- Resolver problemas no linealmente separables (como XOR)")
print("- Aprender representaciones jerárquicas")
print("- Capturar interacciones complejas entre features")
print("\nEsto lo veremos en el próximo notebook con PyTorch.")

---

## Guía Teórico-Conceptual

### 1. Historia y Contexto del Perceptrón

**1943**: McCulloch y Pitts proponen el primer modelo matemático de neurona

**1958**: Frank Rosenblatt construye el Mark I Perceptron en Cornell, una computadora analógica que podía aprender a reconocer patrones visuales

**1960s**: Optimismo inicial. Se pensaba que los perceptrones podrían resolver cualquier problema de reconocimiento de patrones

**1969**: Minsky y Papert publican "Perceptrons", demostrando matemáticamente sus limitaciones (no puede resolver XOR ni problemas no lineales). Esto causó el primer "invierno de la IA"

**1986**: Rumelhart, Hinton y Williams popularizaron backpropagation, permitiendo entrenar redes multicapa y superar las limitaciones del perceptrón simple

**Hoy**: El perceptrón es el building block de redes con millones de neuronas (BERT tiene 110M de parámetros)

### 2. Fundamentos Matemáticos

**Producto punto como similitud:**

El producto punto $w \cdot x$ mide la similitud entre el vector de pesos y el vector de entrada:
- Si son similares (apuntan en la misma dirección): producto alto
- Si son opuestos: producto bajo o negativo
- Si son ortogonales: producto cero

**Interpretación geométrica:**

El perceptrón define un hiperplano en el espacio de features:
$$w_1 x_1 + w_2 x_2 + ... + w_n x_n + b = 0$$

- Un lado del hiperplano: clase 1
- Otro lado: clase 0
- El vector w es perpendicular al hiperplano
- El bias b desplaza el hiperplano del origen

**Convergencia:**

Teorema de convergencia del perceptrón (Rosenblatt, 1958):
> Si los datos son linealmente separables, el algoritmo del perceptrón converge en un número finito de pasos.

Pero: No hay garantía sobre cuántas iteraciones necesitará.

### 3. Relación con Descenso de Gradiente

La regla de actualización del perceptrón es un caso especial de descenso de gradiente:

**Descenso de gradiente general:**
$$w_{nuevo} = w_{viejo} - \alpha \frac{\partial L}{\partial w}$$

**Perceptrón:**
$$w_{nuevo} = w_{viejo} + \alpha \cdot error \cdot x$$

El perceptrón actualiza pesos solo cuando hay error (gradiente implícito de 0 cuando no hay error).

### 4. Comparación: Perceptrón vs. Regresión Logística

| Característica | Perceptrón | Regresión Logística |
|----------------|------------|---------------------|
| Función de activación | Escalón | Sigmoide |
| Salida | Binaria (0 o 1) | Probabilidad (0 a 1) |
| Diferenciable | No | Sí |
| Función de pérdida | Errores de clasificación | Cross-entropy |
| Actualización | Solo cuando hay error | Siempre (proporcional al gradiente) |
| Convergencia | Garantizada si linealmente separable | Siempre converge (convexo) |
| Interpretación | Clasificador duro | Clasificador probabilístico |

### 5. Problema XOR: El Límite de la Linealidad

XOR (OR exclusivo) es el ejemplo clásico de problema no linealmente separable:

```
Entrada (x₁, x₂)  | Salida
(0, 0)            | 0
(0, 1)            | 1  
(1, 0)            | 1
(1, 1)            | 0
```

Si graficamos en 2D:
```
  x₂
  |
1 | 1   0    ← No hay línea recta que separe
  |           los 1s de los 0s
0 | 0   1
  +--------- x₁
    0   1
```

**Solución:** Red multicapa (MLP) con una capa oculta puede resolver XOR al proyectar los datos a un espacio de mayor dimensión donde SÍ son linealmente separables.

### 6. Tasa de Aprendizaje: Finding the Sweet Spot

El hiperparámetro α (learning rate) controla cuánto ajustamos los pesos en cada paso:

**α muy pequeño (ej: 0.001):**
- Pro: Actualizaciones suaves, convergencia estable
- Contra: Aprendizaje muy lento, muchas épocas necesarias

**α muy grande (ej: 10):**
- Pro: Aprendizaje rápido inicialmente
- Contra: Puede oscilar sin converger, inestabilidad

**α óptimo (ej: 0.1 - 0.5):**
- Balance entre velocidad y estabilidad
- Depende del problema y la escala de los datos

**Estrategias modernas:**
- Learning rate decay: Empezar alto y reducir gradualmente
- Optimizadores adaptativos (Adam, RMSprop): Ajustan α automáticamente

### 7. Del Perceptrón al Deep Learning

El camino evolutivo:

1. **Perceptrón simple** (1958): Una neurona, función escalón
2. **Adaline** (1960): Una neurona, función lineal, descenso de gradiente
3. **MLP (Perceptrón Multicapa)** (1986): Múltiples capas, backpropagation
4. **Redes Convolucionales (CNN)** (1998): Para imágenes, LeCun
5. **Redes Recurrentes (RNN/LSTM)** (1997): Para secuencias temporales
6. **Transformers** (2017): Attention, BERT, GPT

**Elemento común:** Todos usan variaciones de la regla básica w = w + Δw que vimos en el perceptrón.

---

## Preguntas y Respuestas para Estudio

### Preguntas Conceptuales

**1. ¿Por qué se dice que el perceptrón es un "clasificador lineal"?**

*Respuesta:* Porque la frontera de decisión que aprende es lineal (una línea en 2D, un hiperplano en dimensiones superiores). La ecuación $w_1x_1 + w_2x_2 + ... + w_nx_n + b = 0$ define un hiperplano que separa las dos clases. Todo lo que esté de un lado se clasifica como 1, del otro como 0.

**2. ¿Qué significa "linealmente separable"?**

*Respuesta:* Un conjunto de datos es linealmente separable si existe un hiperplano que puede seperar perfectamente las dos clases. Por ejemplo, en 2D significa que podemos dibujar una línea recta que deje todos los puntos de una clase a un lado y todos los de la otra clase al otro lado.

**3. ¿Por qué el perceptrón no puede resolver XOR?**

*Respuesta:* Porque XOR no es linealmente separable. Si graficamos los 4 puntos de XOR en 2D, no existe ninguna línea recta que separe correctamente los 1s de los 0s. Necesitaríamos una frontera de decisión curva o quebrada, lo cual requiere una red multicapa.

**4. ¿Cuál es la diferencia entre una época y una iteración?**

*Respuesta:*
- **Iteración**: Una actualización de pesos (procesar una muestra o un batch)
- **Época**: Una pasada completa por todo el dataset de entrenamiento

Si tenemos 100 muestras y procesamos una a la vez, una época = 100 iteraciones.

**5. ¿Por qué inicializamos los pesos aleatoriamente y no en cero?**

*Respuesta:* Si todos los pesos empiezan en cero (o el mismo valor), todas las neuronas aprenderán lo mismo (simetría). La inicialización aleatoria rompe esta simetría, permitiendo que cada neurona especialice en features diferentes. En redes multicapa esto es crítico.

### Preguntas Técnicas

**6. En la regla de actualización w = w + α × error × x, ¿qué pasa si error = 0?**

*Respuesta:* No se actualizan los pesos (w = w + 0 = w). Esto es eficiente: solo ajustamos cuando el modelo se equivoca. En contraste, algoritmos como gradient descent actualizan pesos en cada iteración, incluso cuando la predicción es correcta.

**7. ¿Por qué multiplicamos por x_i en la actualización de pesos?**

*Respuesta:* Es la regla de Hebbian: "neuronas que disparan juntas, se conectan juntas". Si x_i es grande (la feature está presente) y hay error, queremos ajustar mucho ese peso porque esa feature es relevante. Si x_i = 0 (feature ausente), ese peso no se actualiza porque no influyó en la predicción.

**8. ¿Qué representa el bias (sesgo) geométricamente?**

*Respuesta:* El bias desplaza el hiperplano de decisión desde el origen. Sin bias, el hiperplano siempre pasa por (0,0,...,0). El bias permite mover el hiperplano para ajustarse mejor a los datos. Es equivalente a agregar una feature constante = 1.

**9. Si el historial de errores oscila sin bajar, ¿qué podría estar pasando?**

*Respuesta:* Tres posibilidades:
1. **Problema no linealmente separable**: No existe solución perfecta
2. **Learning rate muy alto**: Oscila alrededor del mínimo sin converger
3. **Ruido en los datos**: Ejemplos contradictorios o mal etiquetados

**10. ¿Por qué np.dot(w, x) y no un bucle for?**

*Respuesta:* El producto punto está altamente optimizado en NumPy (usa BLAS, instrucciones SIMD). Es 100-1000x más rápido que un bucle Python. Esta diferencia es crítica en deep learning donde hacemos millones de estas operaciones.

### Preguntas de Aplicación

**11. Si tuvieras un dataset con 3 clases (positivo, negativo, neutral), ¿podrías usar un perceptrón simple?**

*Respuesta:* No directamente. El perceptrón simple es binario. Soluciones:
1. **One-vs-Rest**: Entrenar 3 perceptrones (positivo vs resto, negativo vs resto, neutral vs resto)
2. **Softmax**: Usar una red con múltiples salidas y función softmax (generalización del perceptrón)
3. **Descomposición binaria**: Positivo vs no-positivo, luego negativo vs neutral

**12. ¿En qué casos un perceptrón simple podría ser preferible a una red neuronal profunda?**

*Respuesta:*
- Dataset muy pequeño (< 100 muestras)
- Problema conocidamente lineal
- Necesidad de interpretabilidad extrema (pesos son directamente interpretables)
- Recursos computacionales MUY limitados
- Como baseline rápido antes de modelos complejos

**13. Si el modelo converge en la época 5 pero seguís entrenando 15 épocas más, ¿qué puede pasar?**

*Respuesta:* Con el perceptrón simple, no pasa nada malo. Una vez que error = 0, los pesos dejan de actualizarse. Sin embargo, en redes más complejas, entrenar de más puede causar overfitting. Es buena práctica usar "early stopping": detener cuando la validación deja de mejorar.

**14. ¿Cómo adaptarías este código para procesar el dataset amazon_cells_labelled.txt del Notebook 1?**

*Respuesta:*
```python
# 1. Cargar datos
df = pd.read_csv('amazon_cells_labelled.txt', sep='\t', names=['review', 'sentiment'])

# 2. Construir vocabulario automáticamente
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(binary=True, max_features=50)
X = vectorizer.fit_transform(df['review']).toarray()
y = df['sentiment'].values

# 3. Entrenar (el código del perceptrón es el mismo)
```

**15. Explicá por qué Bag of Words binario pierde información. ¿Qué alternativas hay?**

*Respuesta:* Bag of Words binario solo indica presencia/ausencia, ignorando:
1. **Frecuencia**: "muy muy bueno" se representa igual que "muy bueno"
2. **Orden**: "no me gusta" = "me gusta no"
3. **Contexto**: No captura bi-gramas ni tri-gramas

**Alternativas:**
- **Conteos**: [1, 2, 0, 1] en vez de binario
- **TF-IDF**: Ponderar por importancia en el corpus
- **N-gramas**: Incluir pares/triples de palabras
- **Embeddings**: Word2Vec, GloVe, BERT (representaciones densas)

---

## Ejercicios propuestos

### Ejercicio 1: experimentación con tasa de aprendizaje
Entrená el perceptrón con distintos valores de `tasa_aprendizaje`: 0.01, 0.1, 0.5 y 1.0. Compará la curva de errores por época.

### Ejercicio 2: ampliación del corpus
Agregá más frases positivas y negativas. Después observá dos cosas:

1. Si el perceptrón necesita menos épocas para estabilizarse.
2. Si mejora la pequeña prueba externa.

### Ejercicio 3: vocabulario reducido
Dejá solo 3 o 4 palabras en el vocabulario y analizá cómo cambia la capacidad del modelo para clasificar.

### Ejercicio 4: problema XOR
Intentá entrenar el perceptrón con XOR y registrá en qué sentido falla.

## Cierre

Este notebook no buscó rendimiento, sino comprensión. Lo importante es que ahora podés seguir el recorrido completo de una neurona artificial: entrada, pesos, suma ponderada, activación y actualización por error.

En el próximo cuaderno vamos a conservar esta intuición, pero dejaremos de actualizar todo a mano: pasaremos a una red multicapa con PyTorch.
